# 🍜 頑固ラーメン屋のAIを作ろう (Build a Stubborn Ramen AI)

---
```text
 FFFFF   A   QQQQ      BBBB   OOO  TTTTT
 F      A A  Q  Q      B  B  O   O   T
 FFF   AAAAA Q  Q      BBBB  O   O   T
 F     A   A Q  Q      B  B  O   O   T
 F     A   A  QQ Q     BBBB   OOO    T
```
# 📘 ラボ0：はじめに — 生成AIの基礎と環境準備
### (Lab 0: Introduction — GenAI Basics & Environment Setup)
---

ようこそ！このワークショップでは、**小規模言語モデル（SLM: Small Language Model）** をゼロから構築します。

## 🗓️ 本日のスケジュール (Today's Schedule)

| 時間 | 内容 | ノートブック |
| :--- | :--- | :--- |
| 午前 (1) | 生成AIの基礎・環境確認 | `lab0` (このノートブック) |
| 午前 (2) | 最初の言語モデル (Bigram) を作る | `lab1` |
| 昼休み | 🍜 | |
| 午後 (1) | Transformerへの「脳の移植手術」 | `lab2` 前半 |
| 午後 (2) | **自分のデータ**でAIを訓練する | `lab2` 後半 |
| 午後 (3) | 実用SLM: LoRAファインチューニング + MCPデモ | `lab3` |

## 🎯 本日のゴール (Goals)
1. LLM/SLMの仕組みを **コードレベルで** 理解する
2. Transformerモデルを **ゼロから** 構築・訓練する
3. **自分の業務データ** で動くAIを作り、持ち帰る


---
# 1. すべての始まり：AI・機械学習・深層学習・生成AI

AIという言葉は広い概念です。まず、用語の関係を整理しましょう。

```text
┌─────────────────────────────────────────────┐
│ 人工知能 (AI)                                │
│  ┌────────────────────────────────────────┐ │
│  │ 機械学習 (Machine Learning)             │ │
│  │  ・データからパターンを自動で学習       │ │
│  │  ┌───────────────────────────────────┐ │ │
│  │  │ ディープラーニング (Deep Learning) │ │ │
│  │  │  ・多層ニューラルネットワーク      │ │ │
│  │  │  ┌──────────────────────────────┐ │ │ │
│  │  │  │ 生成AI (Generative AI)        │ │ │ │
│  │  │  │  ・新しいコンテンツを「生成」 │ │ │ │
│  │  │  │  ・LLM / 画像生成 など ★今日★ │ │ │ │
│  │  │  └──────────────────────────────┘ │ │ │
│  │  └───────────────────────────────────┘ │ │
│  └────────────────────────────────────────┘ │
└─────────────────────────────────────────────┘
```

## 機械学習 (ML) とは？
従来のプログラミングでは、人間がルールを一つ一つ書いていました。
機械学習では、コンピュータに **大量のデータ** を与え、データの中から自動的に **パターンやルール** を学習させます。

- **シャローラーニング（伝統的手法）**: 回帰モデルや決定木など。表形式データ（家の価格予測など）に強いが、テキストや画像のような複雑なデータは苦手。
- **ディープラーニング（深層学習）**: 人間の脳の神経回路網にヒントを得た、多くの層を重ねたモデル。複雑で抽象的なパターンを自動で発見できる。現代のAI革命の原動力。

## 生成AI (GenAI) とは？
単にデータを予測・分類するだけでなく、学習したデータと似た **新しいオリジナルのコンテンツを「生成」** できるAIです。
文章・画像・音楽・プログラムコードなどを生成します。

## 大規模言語モデル (LLM) と 小規模言語モデル (SLM)
生成AIの中でも「言葉」を扱うのが **言語モデル** です。

| | LLM (大規模) | SLM (小規模) ★今日作るもの★ |
| :--- | :--- | :--- |
| パラメータ数 | 数百億〜数兆 | 数百万〜数十億 |
| 例 | GPT-4, Claude, Gemini | Qwen 0.5B, TinySwallow など |
| 実行環境 | 大規模データセンター | 手元のPC・社内サーバでも可 |
| 利点 | 高い汎用性 | **低コスト・プライバシー・オンプレ運用** |

企業での実用では、SLMの「自社データで訓練し、自社内で動かせる」という性質が大きな価値を持ちます。


---
# 2. 言語モデルの基本原理：次のトークン予測

LLMもSLMも、やっていることは驚くほどシンプルです。

> **「次に来る文字（トークン）は何か？」をひたすら予測する。**

例えば「今日の天気は」という文が与えられたら、モデルは次に来そうな言葉として「晴れ」の確率が高い、と予測します。

```text
入力: 「今日の天気は」
        │
        ▼
   ┌──────────┐
   │ 言語モデル │  →  次の文字の確率分布を出力
   └──────────┘      「晴」: 45%  「雨」: 30%  「曇」: 20% ...
        │
        ▼
出力: 「晴」を選ぶ → 「今日の天気は晴」 → これを繰り返す
```

この単純な予測を高速で繰り返すだけで、人間が書いたかのような自然な文章が生成されます。
**AIは魔法ではなく、巨大な「確率の計算機」** です。今日はこれを実際に自分の手で作って確かめます。

# 3. モデル構築のパイプライン (The Pipeline)

機械学習モデルの構築は、料理のレシピに似ています。
正しい材料（データ）を、正しい手順で調理することで、良いモデルが完成します。

```text
 a. データ収集  →  b. 前処理      →  c. モデル設計  →  d. トレーニング  →  e. 推論
 (テキスト)       (文字→数字変換)    (Transformer)     (損失を最小化)     (テキスト生成)
```

| 工程 | 内容 | 今日のラボ |
| :--- | :--- | :--- |
| データ | モデルの「教科書」。質と量が性能を決める | lab1, lab2 |
| 前処理 | 文字を数字（トークン）に変換 | lab1 |
| モデル設計 | ネットワーク構造とハイパーパラメータを決定 | lab1, lab2 |
| トレーニング | 予測→誤差計算→修正 のループ | lab1, lab2, lab3 |
| 推論 | 訓練済みモデルで文章を生成 | 各ラボの最後 |


---
# 4. Jupyter Notebook の使い方 (How to Use Jupyter)

今あなたが見ているこの画面が **JupyterLab** です。データサイエンスや機械学習で最も広く使われる対話型開発環境です。

## 画面構成
- **左サイドバー**: ファイルブラウザ（📁）でノートブックやデータファイルを管理
- **メイン作業エリア**: ノートブックがタブ形式で開く（今ここ）
- **上部メニューバー**: File, Edit, Run など

## セル (Cell) の種類
- **Markdownセル**: 説明文（今読んでいるこの文章）
- **Codeセル**: Pythonコード。実行すると結果がすぐ下に表示される

## ⌨️ 最重要ショートカット
| 操作 | キー |
| :--- | :--- |
| **セルを実行して次へ** | `Shift + Enter` ★これだけ覚えればOK★ |
| セルを実行してその場に留まる | `Ctrl + Enter` |
| 上/下にセルを追加 | `Esc` → `A` / `B` |
| カーネル再起動（困ったとき） | メニュー: Kernel → Restart Kernel |

## 🚨 困ったときは
1. コードが止まらない → メニューの ⏹ (Interrupt) ボタン
2. 変数がおかしくなった → **Kernel → Restart Kernel** して、上のセルから順に再実行
3. エラーが直らない → 各ラボの最後にある「🆘 お助けコーナー」の正解コードをコピペ

では、下のセルで練習してみましょう。セルをクリックして `Shift + Enter` を押してください。


In [ ]:
# ✏️ 練習1: このセルを実行してみましょう (Shift + Enter)
print("こんにちは！Jupyter Notebookへようこそ 🍜")

# Pythonは電卓としても使えます
1 + 2 * 3

In [ ]:
# ✏️ 練習2: 変数 (Variables)
# 変数とは「値に名前を付けた箱」です。一度実行すると、
# 後のセルでもその変数を使えます（ノートブックの重要な性質！）

ramen_price = 900          # ラーメンの価格
topping_price = 150        # トッピングの価格

total = ramen_price + topping_price
print(f"合計金額: {total} 円")   # f"..." は文字列に変数を埋め込む書き方（f-string）

---
# 5. 本日使うPythonライブラリ (Python Libraries)

**ライブラリ**とは、便利な機能が詰まった「道具箱」です。他の人が作った複雑な機能を、簡単な命令で呼び出せます。

| ライブラリ | 役割 | 例え |
| :--- | :--- | :--- |
| **PyTorch** (`torch`) | ニューラルネットワーク構築・訓練 | AIの工作キット |
| **NumPy** (`numpy`) | 高速な数値計算・配列操作 | 高性能な電卓 |
| **Matplotlib** | グラフ描画 | グラフ用紙とペン |
| **Transformers / PEFT** | 学習済みモデルの利用・微調整 (lab3) | 完成品AIの改造ツール |

下のセルで、環境がすべて揃っているか確認します。


In [ ]:
# ✅ 環境チェック (Environment Check)
# すべてのライブラリが正しくインストールされているか確認します

import sys
import torch
import numpy as np
import matplotlib

print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
print(f"NumPy   : {np.__version__}")
print(f"Matplotlib : {matplotlib.__version__}")

# GPUが使えるかどうかの確認
# GPUがあると訓練が数十倍速くなります（無くても本日のラボは動きます）
if torch.cuda.is_available():
    print(f"\n🚀 GPU利用可能: {torch.cuda.get_device_name(0)}")
else:
    print("\n💻 GPUなし: CPUモードで実行します（lab1, lab2はCPUでもOK）")

In [ ]:
# ✏️ 練習3: NumPy — AIの土台となる「配列計算」
# AIの内部では、すべてのデータが数値の配列（ベクトル・行列）として扱われます

import numpy as np

# 1週間のラーメン販売数（架空データ）
sales = np.array([120, 135, 98, 150, 180, 210, 195])
days = ["月", "火", "水", "木", "金", "土", "日"]

print(f"合計販売数: {sales.sum()} 杯")
print(f"1日平均   : {sales.mean():.1f} 杯")
print(f"最高記録  : {sales.max()} 杯 ({days[sales.argmax()]}曜日)")

# 全要素への一括演算（ベクトル演算）— ループ不要！
# この「配列まるごと計算」がAI計算の基本です
revenue = sales * 900
print(f"売上（円）: {revenue}")

In [ ]:
# ✏️ 練習4: Matplotlib — データを「見る」
# 数字の羅列より、グラフの方が一目で分かります。
# 本日、モデルの学習の進み具合（Loss曲線）を見るのに使います。

import matplotlib.pyplot as plt
import japanize_matplotlib  # 日本語表示用（エラーが出たら次のセルを実行）

plt.figure(figsize=(8, 4))
plt.bar(days, sales, color="salmon")
plt.title("ラーメン販売数（1週間）")
plt.xlabel("曜日")
plt.ylabel("販売数（杯）")
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
# 🆘 上のセルで japanize_matplotlib のエラーが出た場合のみ、このセルを実行してください
# （インストール後、上のセルを再実行）
%pip install -q japanize-matplotlib

In [ ]:
# ✏️ 練習5: PyTorchのテンソル (Tensor)
# テンソルとは、AI計算に特化した高性能な多次元配列です。
# NumPyの配列とほぼ同じですが、GPU計算と「自動微分」（学習に必須）に対応しています。
# 本日のモデルの中を流れるデータは、すべてこのテンソルです。

import torch

t = torch.tensor([1, 2, 3, 4, 5])
print(f"テンソル      : {t}")
print(f"形状 (shape)  : {t.shape}")     # テンソルの「サイズ」。デバッグで最も見る情報！
print(f"データ型      : {t.dtype}")

# 2次元テンソル（行列）— 本日 (バッチ, 文字数) の形で頻出します
m = torch.zeros((3, 4))   # 3行4列のゼロ行列
print(f"\n2次元テンソルの形状: {m.shape}  ← (行, 列)")

---
# ✅ ラボ0 完了！

これで準備は万端です。確認しましょう：

- [x] AI → ML → DL → 生成AI → LLM/SLM の関係を理解した
- [x] 言語モデルの正体は「次の文字の確率計算機」だと知った
- [x] Jupyter Notebookのセル実行 (`Shift + Enter`) ができる
- [x] PyTorch・NumPy・Matplotlibが動くことを確認した

次は **`lab1_morning_bigram.ipynb`** を開いてください。
いよいよ、最初の言語モデル「頑固ラーメン屋のAI」を作ります！🍜
